In [ ]:
import mlflow

from mlflow.models import infer_signature
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

In [ ]:
# -----------------------------
# 1. Load & prepare data
# -----------------------------
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pack test data into one DataFrame (mlflow.evaluate expects this format)
test_data = X_test.copy()
test_data["target"] = y_test

In [ ]:
# -----------------------------
# 2. Setup MLflow experiment
# -----------------------------
mlflow.set_experiment("diabetes_with_evaluate")

In [ ]:
# -----------------------------
# 3. Train, log & evaluate
# -----------------------------
with mlflow.start_run(run_name="rf_with_mlflow_evaluate") as run:
    
    # Model training
    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(X_train, y_train)

    # Log model
    signature = infer_signature(X_train, model.predict(X_train))
    
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="rf_model",
        signature=signature,
        input_example=X_train.head(1),
        registered_model_name="DiabetesRandomForestModel"
    )

    # -----------------------------
    # 4. Evaluate model with mlflow.evaluate
    # -----------------------------
    results = mlflow.models.evaluate(
        model=model_info.model_uri,         # can also be "runs:/<run_id>/model"
        data=test_data,
        targets="target",
        model_type="regressor",             # regressor/classifier
        evaluators=["default"],             # default includes regression metrics
    )

    print("Evaluation metrics:")
    print(results.metrics)
    print("Artifacts logged:", results.artifacts.keys())

    print(f"Run ID: {run.info.run_id}")
